# LUDB — download, convert, label, visualize

[Lobachevsky University ECG Database](https://physionet.org/content/ludb/1.0.1/) (200 × 10 s, 12-lead, 500 Hz).

LUDB is the only public set here with **cardiologist-drawn P / QRS / T boundaries on every lead** plus a wide diagnostic taxonomy (rhythm, axis, conduction, extrasystoles, hypertrophy, pacing, ischemia/STEMI, nonspecific repolarization).

Outputs (skip files that already exist):

- raw WFDB → `data/evaluation/raw/ludb/`
- 12-lead `.npy` + Graph_Visualizer `.pkl` → `data/evaluation/processed/ludb/signals/`
- per-record diagnosis **and** per-lead delineation JSON → `data/evaluation/processed/ludb/labels/`

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd() if (Path.cwd() / "evaluation" / "common.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "evaluation"))

import numpy as np
import pandas as pd
import wfdb
from IPython.display import display

import common as C

# --- subset ---
N_RECORDS = 20          # set None for all 200 (dataset is only ~24 MB)
RECORD_IDS = None       # e.g. ["1", "33", "50"] to pin specific records
REQUIRE_ISCHEMIA = False
OVERWRITE_PROCESSED = False
SEED = 42

RAW_DIR, PROC_DIR = C.dataset_dirs("ludb")
FALLBACK = [REPO / "data" / "LU_DB"]  # reuse files already on disk
print("raw:", RAW_DIR)
print("processed:", PROC_DIR)

## 1. Catalogue + subset

In [ ]:
ludb_csv = RAW_DIR / "ludb.csv"
status = C.download_files(
    "ludb",
    ["ludb.csv", "RECORDS", "README"],
    RAW_DIR,
    fallback_dirs=FALLBACK,
    version="1.0.1",
)
print("metadata:", C.summarize_status(status))

catalogue = pd.read_csv(ludb_csv)
catalogue.columns = [c.strip() for c in catalogue.columns]
catalogue["ID"] = catalogue["ID"].astype(str).str.strip()
print(f"Catalogue rows: {len(catalogue)}")
display(catalogue.head())

pool = catalogue.copy()
if REQUIRE_ISCHEMIA:
    pool = pool[pool["Ischemia"].fillna("").astype(str).str.strip().ne("")]

if RECORD_IDS:
    wanted = {str(x).lstrip("0") for x in RECORD_IDS}
    selected = pool[pool["ID"].isin(wanted)]
else:
    n = len(pool) if N_RECORDS is None else min(int(N_RECORDS), len(pool))
    selected = pool.sample(n=n, random_state=SEED).sort_values("ID")

record_ids = [str(i) for i in selected["ID"].tolist()]
print(f"Selected {len(record_ids)} records: {record_ids}")

## 2. Download WFDB (dat/hea + 12 lead annotation files)

In [ ]:
files = []
for rid in record_ids:
    files.extend(C.ludb_record_files(rid))

status = C.download_files(
    "ludb", files, RAW_DIR, fallback_dirs=FALLBACK, version="1.0.1"
)
print("waveforms:", C.summarize_status(status))
failed = [k for k, v in status.items() if v == "failed"]
if failed:
    print("failed:", failed[:12], ("..." if len(failed) > 12 else ""))

## 3. Convert to `.npy` + labels

In [ ]:
index_rows = []
for rid in record_ids:
    if C.processed_exists(PROC_DIR, rid) and not OVERWRITE_PROCESSED:
        print(f"skip processed {rid}")
        continue

    rec_path = RAW_DIR / "data" / rid
    if not rec_path.with_suffix(".hea").is_file():
        rec_path = FALLBACK[0] / "data" / rid
    record = wfdb.rdrecord(str(rec_path))
    signal_12, channels = C.to_12_lead(record.p_signal.T, record.sig_name)
    C.save_signal_pair(PROC_DIR / "signals" / rid, signal_12, record.fs, channels)

    parsed = C.parse_ludb_comments(record.comments)
    delineation = C.load_ludb_delineation(rec_path)
    n_qrs = sum(len(v.get("R_peaks", [])) for v in delineation.values())
    payload = {
        "dataset": "ludb",
        "record_id": rid,
        "fs": int(record.fs),
        "n_samples": int(record.sig_len),
        "channels": channels,
        "age": parsed["age"],
        "sex": parsed["sex"],
        "diagnoses": parsed["diagnoses"],
        "delineation": delineation,
        "n_qrs_annotations_all_leads": n_qrs,
    }
    C.save_label_json(PROC_DIR / "labels" / f"{rid}.json", payload)

    dx = parsed["diagnoses"]
    index_rows.append({
        "record_id": rid,
        "fs": int(record.fs),
        "n_samples": int(record.sig_len),
        "age": parsed["age"],
        "sex": parsed["sex"],
        "rhythm": "; ".join(dx["Rhythm"]),
        "axis": "; ".join(dx["Electric axis of the heart"]),
        "conduction": "; ".join(dx["Conduction abnormalities"]),
        "extrasystoles": "; ".join(dx["Extrasystolies"]),
        "hypertrophy": "; ".join(dx["Hypertrophies"]),
        "pacing": "; ".join(dx["Cardiac pacing"]),
        "ischemia": "; ".join(dx["Ischemia"]),
        "nonspecific_repol": "; ".join(dx["Non-specific repolarization abnormalities"]),
        "other": "; ".join(dx["Other states"]),
        "n_qrs_annotations_all_leads": n_qrs,
        "npy": str((PROC_DIR / "signals" / f"{rid}.npy").relative_to(C.REPO_ROOT)),
    })
    print(f"wrote {rid}  ischemia={bool(dx['Ischemia'])}  qrs_marks={n_qrs}")

C.write_index(PROC_DIR, index_rows)
index = pd.read_csv(PROC_DIR / "index.csv")
print(f"index rows: {len(index)}")
display(index.head())

## 4. Visualizations

LUDB labels are **multi-label diagnoses** plus **time-aligned wave marks**. Plots: 12-lead strip, annotated lead II, category prevalence, diagnosis co-occurrence.

In [ ]:
import json
import pickle
from matplotlib import pyplot as plt

index = pd.read_csv(PROC_DIR / "index.csv")
example_id = str(index.iloc[0]["record_id"])
sig = np.load(PROC_DIR / "signals" / f"{example_id}.npy")
with open(PROC_DIR / "signals" / f"{example_id}.pkl", "rb") as f:
    pickle_meta = pickle.load(f)
labels = json.loads((PROC_DIR / "labels" / f"{example_id}.json").read_text(encoding="utf-8"))

fig, ax = C.plot_12_lead_strip(
    sig, pickle_meta["fs"], pickle_meta["channels"],
    title=f"LUDB {example_id}  |  {labels.get('sex')}, {labels.get('age')}y  |  {'; '.join(labels['diagnoses'].get('Rhythm') or ['?'])}",
)
C.save_figure(fig, PROC_DIR, f"ludb_{example_id}_12lead")
display(fig)
plt.close(fig)

fig, ax = C.plot_ludb_delineation(
    sig, pickle_meta["fs"], pickle_meta["channels"], labels["delineation"],
    lead="II",
    title=f"LUDB {example_id} lead II — cardiologist P / QRS / T",
)
C.save_figure(fig, PROC_DIR, f"ludb_{example_id}_delineation")
display(fig)
plt.close(fig)

In [ ]:
fields = [
    ("rhythm", "Rhythm"),
    ("axis", "Axis"),
    ("conduction", "Conduction"),
    ("extrasystoles", "Extrasystoles"),
    ("hypertrophy", "Hypertrophy"),
    ("pacing", "Pacing"),
    ("ischemia", "Ischemia"),
    ("nonspecific_repol", "Nonspecific repol."),
    ("other", "Other"),
]
present = pd.Series(
    {title: int(index[col].fillna("").astype(str).str.strip().ne("").sum()) for col, title in fields}
)
fig, ax = C.plot_category_counts(present, title="LUDB subset — records with each diagnostic family")
C.save_figure(fig, PROC_DIR, "ludb_category_counts")
display(fig)
plt.close(fig)

binary = pd.DataFrame({
    title: index[col].fillna("").astype(str).str.strip().ne("").astype(int)
    for col, title in fields
})
fig, ax = C.plot_cooccurrence(binary, title="LUDB subset — co-occurrence of diagnostic families")
C.save_figure(fig, PROC_DIR, "ludb_cooccurrence")
display(fig)
plt.close(fig)

print("Figures →", PROC_DIR / "figures")